<a href="https://colab.research.google.com/github/filipsajtlava/dspracticum2025-tismaci/blob/homework7/homeworks/hw7/homework07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.3 MB/s eta 0:00:00


In [4]:
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

In [5]:
train_data_path = "http://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_train.csv"
test_data_path = "https://raw.githubusercontent.com/filipsajtlava/dspracticum2025-tismaci/refs/heads/master/homeworks/hw7/appartments_test.csv"

In [6]:
try:
    df_train = pd.read_csv(train_data_path)
    df_test = pd.read_csv(test_data_path)
    print("Data byla načtena.")
except Exception as e:
    print(f"Chyba: {e}")

Data byla načtena.


In [7]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator', 'address']
not_using = ['text', 'first_seen', 'last_seen', 'price']
using = [col for col in df_train.columns if col not in not_using]

X_train = df_train[using].copy()
y_train = df_train['price'].copy()
X_test = df_test[using].copy()

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')


In [8]:
model = CatBoostRegressor(
    iterations=1500,
    learning_rate=0.04,
    depth=7,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=300,
    cat_features=categorical
)

In [9]:
model.fit(X_train, y_train, early_stopping_rounds=300)

0:	learn: 0.2713943	total: 68.3ms	remaining: 1m 42s
300:	learn: 0.2131558	total: 5.76s	remaining: 22.9s
600:	learn: 0.2040541	total: 12.6s	remaining: 18.9s
900:	learn: 0.1992164	total: 17.8s	remaining: 11.8s
1200:	learn: 0.1947655	total: 24.6s	remaining: 6.12s
1499:	learn: 0.1915802	total: 29.7s	remaining: 0us


In [10]:
feature_importances = pd.Series(model.get_feature_importance(), index=X_train.columns)
top_features = feature_importances.sort_values(ascending=False).head(15)
print(f'Dulezite promenne: {top_features}.')
flop_features = feature_importances[feature_importances < 0.1]
if not flop_features.empty:
    print(f"Málo důležité proměnné ({len(flop_features)}):")
    print(flop_features)

Dulezite promenne: area                               24.428789
layout                             16.705653
construction                       14.721440
condition                          14.603320
ownership                           8.650047
gps_lon                             5.190385
elevator                            3.081283
balcony_area                        3.009621
gps_lat                             2.448742
poi_doctors_nearest                 1.627158
poi_school_kindergarten_nearest     0.953655
id                                  0.938376
poi_leisure_time_nearest            0.831178
poi_grocery_nearest                 0.816375
poi_restaurant_nearest              0.773172
dtype: float64.
Málo důležité proměnné (9):
address                          0.000000
garden_area                      0.000000
parking                          0.094121
poi_doctors_count                0.000000
poi_leisure_time_count           0.057813
poi_school_kindergarten_count    0.078964
poi_transp

In [11]:
categorical = ['layout', 'construction', 'condition', 'ownership', 'elevator']
not_using = ['text', 'first_seen', 'last_seen', 'price']
flop = flop_features.index
flop = flop.tolist()
using = [col for col in df_train.columns if col not in not_using + flop]

X_train = df_train[using].copy()
y_train = df_train['price'].copy()
X_test = df_test[using].copy()

for col in categorical:
    if col in X_train.columns:
        X_train[col] = X_train[col].fillna('N/A').astype('str')
        X_test[col] = X_test[col].fillna('N/A').astype('str')

In [14]:
model = CatBoostRegressor(
    iterations=40000,
    learning_rate=0.015,
    depth=8,
    loss_function='MAPE',
    eval_metric='MAPE',
    random_seed=128,
    verbose=500,
    cat_features=categorical
)

In [15]:
model.fit(X_train, y_train, early_stopping_rounds=300)

0:	learn: 0.2717949	total: 52.1ms	remaining: 34m 44s
500:	learn: 0.2031127	total: 13.7s	remaining: 18m 1s
1000:	learn: 0.1850572	total: 26.4s	remaining: 17m 6s
1500:	learn: 0.1777238	total: 38.6s	remaining: 16m 30s
2000:	learn: 0.1718045	total: 50.9s	remaining: 16m 6s
2500:	learn: 0.1668297	total: 1m 3s	remaining: 15m 46s
3000:	learn: 0.1625196	total: 1m 15s	remaining: 15m 34s
3500:	learn: 0.1589359	total: 1m 28s	remaining: 15m 23s
4000:	learn: 0.1558358	total: 1m 41s	remaining: 15m 13s
4500:	learn: 0.1524344	total: 1m 54s	remaining: 15m 1s
5000:	learn: 0.1499785	total: 2m 7s	remaining: 14m 49s
5500:	learn: 0.1472296	total: 2m 19s	remaining: 14m 37s
6000:	learn: 0.1453350	total: 2m 32s	remaining: 14m 25s
6500:	learn: 0.1428071	total: 2m 45s	remaining: 14m 13s
7000:	learn: 0.1403880	total: 2m 58s	remaining: 14m 1s
7500:	learn: 0.1387792	total: 3m 11s	remaining: 13m 48s
8000:	learn: 0.1359723	total: 3m 24s	remaining: 13m 36s
8500:	learn: 0.1338726	total: 3m 37s	remaining: 13m 24s
9000:	l

In [16]:
y_pred_test = model.predict(X_test)

In [17]:
prediction = pd.DataFrame({
    'id': df_test['id'],
    'price': y_pred_test.round(0).astype(int)
})
prediction.to_csv('predikce.csv')